<p align="center">
  <img src="https://i0.wp.com/www.tiempodecine.co/web/wp-content/uploads/2015/11/Robert-De-Niro-in-Taxi-Driver-1976.jpg?resize=750%2C375&ssl=1" style="width:100%; max-width:900px; height:180px; object-fit:cover; border-radius:10px;"/>
</p>
<div style="text-align:center;">
  <h1 style="color:#FFD700; display:inline-block; margin:0;">Optimización del Transporte en Nueva York</h1>
  <p>
    <b>Green Taxi | Machine Learning & Data Science | CRISP-DM</b><br>
    <span style="font-size:1.1em;">Análisis y predicción de tarifas y duración de viajes usando datos reales de taxis verdes de NYC.</span>
  </p>
</div>

# FASE 1: Business Understanding (CRISP-DM)

## Propósito de la Fase de Comprensión del Negocio

En esta primera fase del proceso CRISP-DM, nos enfocamos en entender los objetivos y requisitos del negocio relacionados con el proyecto de análisis de datos. Esto incluye la identificación de los problemas clave que se desean resolver, la definición de los objetivos del proyecto y la comprensión del contexto empresarial en el que se aplicarán los resultados del análisis. El objetivo es asegurar que el proyecto esté alineado con las necesidades del negocio y que los resultados sean relevantes y útiles para la toma de decisiones.

# Carga de Librerías

In [1]:
# Verifica si las librerías necesarias ya están instaladas
try:
    # Ignorar warnings
    import warnings
    warnings.filterwarnings('ignore')
    # Librerías del sistema
    import sys, subprocess
    # Importaciones base
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    # Librerías de sistema de archivos
    import os as os  
    from os import path
    import pickle as pkl
    import joblib
    import duckdb
    # Librerías para muestreo balanceado avanzado
    from sklearn.cluster import KMeans
    from imblearn.under_sampling import ClusterCentroids
    from sklearn.preprocessing import StandardScaler

except ImportError:

    print("Dependencias no encontradas. Instalando ahora...")
    
    # Se ejecutan los comandos de instalación
    %pip install --quiet matplotlib
    %pip install --quiet seaborn
    %pip install --quiet joblib
    %pip install --quiet scipy
    %pip install --quiet duckdb
    %pip install --quiet pyarrow
    %pip install --quiet fastparquet
    %pip install --quiet scikit-learn
    %pip install --quiet imbalanced-learn
    
    print("Instalación completada.")
    
    # Reimportar librerías después de instalación
    from sklearn.cluster import KMeans
    from imblearn.under_sampling import ClusterCentroids
    from sklearn.preprocessing import StandardScaler

## **Contexto del Negocio**

La Comisión de Taxis y Limusinas de Nueva York (TLC) gestiona uno de los sistemas de transporte urbano más complejos del mundo, supervisando:
- **Taxis Amarillos**: Operan en toda la ciudad, principalmente por street-hail
- **Taxis Verdes**: Operan principalmente en boroughs exteriores
- **Vehículos de Alquiler (FHV)**: Servicios previamente arreglados
- **Vehículos de Alto Volumen (HVFHV)**: Servicios de ridesharing como Uber, Lyft

La ciudad enfrenta desafíos críticos de movilidad urbana donde la **distribución desigual de la demanda** genera ineficiencias operativas, afectando tanto a proveedores de servicio como a usuarios finales.

---

## **Problema Central**

### **Desbalance Geográfico y Temporal de la Demanda**

**Evidencias Identificadas:**
- Concentración masiva de viajes en zonas específicas durante horas pico
- Subutilización de flotas en áreas periféricas y horarios valle
- Saturación en distritos financieros y comerciales vs. escasez en zonas residenciales
- Ineficiencia en la asignación de diferentes tipos de servicio (Yellow vs. Green Taxis)

**Impacto en el Negocio:**
- Tiempos de espera excesivos para pasajeros en zonas saturadas
- Bajos ingresos para conductores en zonas de baja demanda
- Congestión vehicular en áreas críticas
- Experiencia de usuario inconsistente

---

## **Objetivos de Negocio**

### **Objetivo Principal**
**Optimizar la distribución espacial y temporal de los servicios de transporte** mediante segmentación inteligente de zonas urbanas.

### **Objetivos Específicos**

1. **Segmentación por Patrones de Demanda**
   - Identificar clusters de zonas con comportamientos similares
   - Categorizar áreas según volumen, temporalidad y tipo de servicio

2. **Gestión de Capacidad Dinámica**
   - Predecir demandas pico y asignar recursos proactivamente
   - Balancear flotas entre zonas complementarias

3. **Optimización de Servicios Especializados**
   - Asignar tipos de vehículos según características zonales
   - Personalizar estrategias por segmento identificado

4. **Planificación de Infraestructura**
   - Identificar ubicaciones óptimas para estaciones de vehículos
   - Optimizar rutas y puntos de espera

---

## **Preguntas Clave de Negocio**

### **1. ¿Cómo podemos segmentar las zonas de NYC según patrones de demanda?**
- *Enfoque:* Identificar clusters naturales basados en volumen horario, tipo de servicio y características temporales
- *Métrica:* Grupos homogéneos con patrones de demanda similares
- *Impacto:* Estrategias diferenciadas por tipo de zona

### **2. ¿Qué zonas presentan mayor saturación y en qué horarios?**
- *Enfoque:* Análisis de concentración de viajes por LocationID y banda horaria
- *Métrica:* Porcentaje de viajes en horas pico vs. capacidad estimada
- *Impacto:* Redistribución prioritaria de recursos

### **3. ¿Existen diferencias significativas en el uso de Yellow vs. Green Taxis por zona?**
- *Enfoque:* Comparativa de distribución geográfica por tipo de servicio
- *Métrica:* Ratio de utilización y patrones de viaje diferenciados
- *Impacto:* Optimización de flotas especializadas

### **4. ¿Qué patrones temporales (horarios, días) definen la demanda por zona?**
- *Enfoque:* Análisis de series temporales y estacionalidad por LocationID
- *Métrica:* Patrones recurrentes y variabilidad horaria/semanal
- *Impacto:* Programación predictiva de recursos

### **5. ¿Cómo se relacionan las zonas de pickup y dropoff en términos de demanda complementaria?**
- *Enfoque:* Análisis de flujos origen-destino y correlaciones espaciales
- *Métrica:* Matrices de transición y zonas con demanda balanceada
- *Impacto:* Estrategias de reposicionamiento eficiente

---

## **Alcance del Proyecto**

### **Dataset Aplicable:**
- **Yellow Taxi**: Análisis principal por cobertura citywide
- **Green Taxi**: Complementario para boroughs exteriores
- **FHV/HVFHV**: Validación cruzada de patrones identificados

### **Criterios de Éxito:**
- Reducción del 15% en tiempos de espera en zonas saturadas
- Aumento del 10% en utilización de flotas en zonas subutilizadas
- Mejora del 20% en balance entre oferta y demanda
- Segmentación clara con al menos 5-7 clusters interpretables

### **Entregables:**
- Modelo de clustering con segmentación de zonas
- Dashboard de monitoreo de demanda en tiempo cuasi-real
- Recomendaciones estratégicas por tipo de zona
- Plan de implementación gradual por prioridad

---

# Carga Yellow Tripdata

In [2]:
# Carga del dataset
yellow_trip = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet")
# Crear una nueva columna con el color del taxi taxi_color = 'green'
yellow_trip['taxi_color'] = 'yellow'
# Crear columna con inicial del color del taxi
yellow_trip['taxi_color_ini'] = yellow_trip['taxi_color'].map({'yellow': 'Y'})
# Agregar columna binaria para yellow and green taxis . map({'yellow': 1, 'green': 0})
yellow_trip['is_color'] = yellow_trip['taxi_color'].map({'yellow': 1, 'green': 0})

# Carga Green Tripdata

In [3]:
# Carga del dataset parquet
green_trip = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-07.parquet")
# Crear una nueva columna con el color del taxi taxi_color = 'green'
green_trip['taxi_color'] = 'green'
# Crear columna con inicial del color del taxi
green_trip['taxi_color_ini'] = green_trip['taxi_color'].map({'green': 'G'})
# Agregar columna binaria para yellow and green taxis . map({'yellow': 1, 'green': 0})
green_trip['is_color'] = green_trip['taxi_color'].map({'yellow': 1, 'green': 0})

---
# **Estrategia de Tratamiento de Datos**

El notebook ejecuta un proceso ETL en dos flujos paralelos para preparar los datos de Taxis y FHVHV.

Para los Taxis (Yellow/Green) , se realizó una normalización de esquema, unificando los nombres de las columnas de fechas (tpep_/lpep_ a pickup_datetime) y alineando los dataframes. Para corregir el desbalance de clases extremo (3.9M vs 48k) , se aplicó random undersampling al dataset Yellow, reduciéndolo al tamaño exacto del Green. Para los FHVHV, un pipeline basado en duckdb muestreó al 1% tres archivos Parquet mensuales (Jul a Sep). Finalmente, ambos datasets procesados (df_taxis_equilibrado y df_fhvhv_sample) se guardaron en nuevos archivos Parquet.

### Unifica nombres de columnas y asegura un esquema consistente entre taxis Yellow and Green.

In [4]:
# Normalización de columnas entre taxis amarillos y verdes
print("\nNormalizando columnas clave de datetime y esquema entre Yellow y Green...")

def alinear_esquema_taxis(df_yellow: pd.DataFrame, df_green: pd.DataFrame):
    """Unifica nombres de columnas y asegura un esquema consistente entre taxis amarillos y verdes."""
    # Renombrar columnas de fecha para que compartan identificadores consistentes
    yellow_renombrado = df_yellow.rename(
        columns={
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime"
        }
    ).copy()
    green_renombrado = df_green.rename(
        columns={
            "lpep_pickup_datetime": "pickup_datetime",
            "lpep_dropoff_datetime": "dropoff_datetime"
        }
    ).copy()
    
    # Conjunto unificado de columnas sin duplicados
    columnas_unificadas = sorted(set(yellow_renombrado.columns).union(set(green_renombrado.columns)))
    
    # Reindexa para garantizar que ambos DataFrames compartan el mismo orden de columnas
    yellow_alineado = yellow_renombrado.reindex(columns=columnas_unificadas)
    green_alineado = green_renombrado.reindex(columns=columnas_unificadas)
    
    # Reportar columnas exclusivas de cada dataset para trazabilidad
    columnas_solo_yellow = sorted(set(yellow_renombrado.columns) - set(green_renombrado.columns))
    columnas_solo_green = sorted(set(green_renombrado.columns) - set(yellow_renombrado.columns))
    
    if columnas_solo_yellow:
        print(f"   Columnas exclusivas Yellow ({len(columnas_solo_yellow)}): {columnas_solo_yellow}")
    if columnas_solo_green:
        print(f"   Columnas exclusivas Green ({len(columnas_solo_green)}): {columnas_solo_green}")
    
    return yellow_alineado, green_alineado, columnas_unificadas

# DataFrames alineados listos para muestreo equilibrado
yellow_trip_aligned, green_trip_aligned, columnas_unificadas = alinear_esquema_taxis(yellow_trip, green_trip)
print(f"   Columnas unificadas: {len(columnas_unificadas)}")
print(f"   Yellow columnas: {len(yellow_trip_aligned.columns)} | Green columnas: {len(green_trip_aligned.columns)}")


Normalizando columnas clave de datetime y esquema entre Yellow y Green...
   Columnas exclusivas Yellow (1): ['Airport_fee']
   Columnas exclusivas Green (2): ['ehail_fee', 'trip_type']
   Columnas unificadas: 25
   Yellow columnas: 25 | Green columnas: 25
   Columnas exclusivas Yellow (1): ['Airport_fee']
   Columnas exclusivas Green (2): ['ehail_fee', 'trip_type']
   Columnas unificadas: 25
   Yellow columnas: 25 | Green columnas: 25


### Balanceo Avanzado: ClusterCentroids + Estratificación Geográfica (50/50)

**Metodología Migrada:** Se ha implementado un algoritmo de muestreo avanzado que combina:

- **ClusterCentroids**: Muestreo inteligente basado en centroides para preservar la estructura de los datos
- **Estratificación Geográfica**: Uso de KMeans para identificar regiones geográficas representativas
- **Balance 50/50**: Garantiza proporciones perfectas para algoritmos de ML
- **Preservación de Patrones**: Mantiene diversidad espacial y temporal en la muestra

**Ventajas sobre Muestreo Aleatorio Simple:**
1. Preserva la distribución geográfica natural de los datos
2. Mantiene patrones de comportamiento por zona
3. Reduce el sesgo introducido por muestreo puramente aleatorio
4. Optimiza la representatividad para clustering posterior

In [5]:
# ADVANCED BALANCED SAMPLING: CLUSTERCENTROIDS + ESTRATIFICACIÓN GEOGRÁFICA
print("\n" + "=" * 80)
print("ADVANCED BALANCED SAMPLING: CLUSTERCENTROIDS + ESTRATIFICACIÓN GEOGRÁFICA")
print("=" * 80)

def create_advanced_balanced_sample_50_50(df_yellow, df_green):
    """
    Crea un dataset balanceado 50/50 usando ClusterCentroids con estratificación geográfica.
    
    METODOLOGÍA:
    1. KMeans clustering para identificar regiones geográficas representativas
    2. ClusterCentroids para muestreo inteligente que preserva la distribución espacial
    3. Estratificación por zona para mantener diversidad geográfica
    4. Balance final 50/50 entre Yellow y Green
    
    Args:
        df_yellow: Yellow taxi DataFrame aligned
        df_green: Green taxi DataFrame aligned
    
    Returns:
        tuple: (yellow_advanced_sample, green_advanced_sample) balanced 50/50
    """
    
    print("INICIANDO MUESTREO AVANZADO CON CLUSTERCENTROIDS...")
    
    # Copiar datasets
    df_yellow = df_yellow.copy()
    df_green = df_green.copy()
    
    # FASE 1: ANÁLISIS INICIAL
    total_yellow_orig = len(df_yellow)
    total_green_orig = len(df_green)
    
    print("ANÁLISIS INICIAL:")
    print(f"   Yellow original: {total_yellow_orig:,} registros")
    print(f"   Green original:  {total_green_orig:,} registros")
    print(f"   Ratio original:  {total_yellow_orig/total_green_orig:.1f}:1 (Yellow:Green)")
    
    # Determinar el tamaño objetivo (50/50)
    # Usamos el tamaño del dataset menor como base para mantener toda la información posible
    target_size_per_class = min(total_yellow_orig, total_green_orig)
    
    print(f"\nOBJETIVO DE BALANCEO 50/50:")
    print(f"   Target por clase: {target_size_per_class:,} registros")
    print(f"   Total final:      {target_size_per_class * 2:,} registros")
    
    # FASE 2: PREPARACIÓN DE FEATURES GEOGRÁFICAS
    print(f"\nFASE 2: EXTRACCIÓN DE FEATURES GEOGRÁFICAS...")
    
    def extract_geographic_features(df):
        """Extrae características geográficas para clustering espacial"""
        features = pd.DataFrame()
        
        # Features básicas de ubicación
        features['PULocationID'] = df['PULocationID'].fillna(0)
        features['DOLocationID'] = df['DOLocationID'].fillna(0)
        
        # Features derivadas si existen columnas de tiempo
        if 'pickup_datetime' in df.columns:
            pickup_dt = pd.to_datetime(df['pickup_datetime'])
            features['pickup_hour'] = pickup_dt.dt.hour
            features['pickup_day_of_week'] = pickup_dt.dt.dayofweek
        else:
            # Features por defecto si no hay tiempo
            features['pickup_hour'] = 12  # Hora promedio
            features['pickup_day_of_week'] = 2  # Día promedio
            
        # Features de distancia/viaje si existen
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in ['trip_distance', 'fare_amount', 'total_amount']:
            if col in numeric_cols:
                features[col] = df[col].fillna(df[col].median() if not df[col].isna().all() else 0)
        
        return features
    
    # Extraer features para ambos datasets
    yellow_features = extract_geographic_features(df_yellow)
    green_features = extract_geographic_features(df_green)
    
    print(f"Features extraídas: {yellow_features.shape[1]} columnas")
    print(f"Yellow features: {yellow_features.shape}")
    print(f"Green features: {green_features.shape}")
    # FASE 3: ESTRATIFICACIÓN GEOGRÁFICA CON KMEANS
    print(f"\nFASE 3: ESTRATIFICACIÓN GEOGRÁFICA CON KMEANS...")
    
    # Combinar features para entrenar clustering geográfico
    combined_features = pd.concat([yellow_features, green_features], ignore_index=True)
    
    # Escalado de features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(combined_features.fillna(0))
    
    # KMeans para identificar regiones geográficas
    n_geographic_clusters = min(10, len(combined_features) // 100)  # Máximo 10 clusters o 1 por cada 100 registros
    n_geographic_clusters = max(3, n_geographic_clusters)  # Mínimo 3 clusters
    
    print(f"   Clusters geográficos: {n_geographic_clusters}")
    
    kmeans_geo = KMeans(n_clusters=n_geographic_clusters, random_state=42, n_init=10)
    geographic_clusters = kmeans_geo.fit_predict(features_scaled)
    
    # Asignar clusters a datasets originales
    yellow_clusters = geographic_clusters[:len(yellow_features)]
    green_clusters = geographic_clusters[len(yellow_features):]
    
    df_yellow['geo_cluster'] = yellow_clusters
    df_green['geo_cluster'] = green_clusters
    
    print(f"   Distribución clusters Yellow: {np.bincount(yellow_clusters)}")
    print(f"   Distribución clusters Green: {np.bincount(green_clusters)}")
    
    # FASE 4: MUESTREO ESTRATIFICADO CON CLUSTERCENTROIDS
    print(f"\nFASE 4: MUESTREO ESTRATIFICADO CON CLUSTERCENTROIDS...")
    
    def stratified_cluster_sampling(df, features, target_size, class_name):
        """Muestreo estratificado usando ClusterCentroids por cluster geográfico"""
        
        if len(df) <= target_size:
            print(f"   {class_name}: Dataset ya es menor que target, manteniendo completo")
            return df
        
        sampled_dfs = []
        
        # Muestreo proporcional por cluster geográfico
        for cluster_id in df['geo_cluster'].unique():
            cluster_mask = df['geo_cluster'] == cluster_id
            cluster_df = df[cluster_mask].copy()
            cluster_features = features[cluster_mask]
            
            # Calcular tamaño proporcional para este cluster
            cluster_proportion = len(cluster_df) / len(df)
            cluster_target = max(1, int(target_size * cluster_proportion))
            cluster_target = min(cluster_target, len(cluster_df))  # No más del disponible
            
            if len(cluster_df) > cluster_target and len(cluster_df) > 1:
                # Usar ClusterCentroids solo si tenemos suficientes puntos
                try:
                    # ClusterCentroids requiere características numéricas
                    cluster_features_clean = cluster_features.fillna(0)
                    
                    # Crear labels artificiales para ClusterCentroids (todos iguales ya que es un solo cluster)
                    y_artificial = np.zeros(len(cluster_features_clean))
                    
                    # ClusterCentroids con ratio específico
                    cluster_centroids = ClusterCentroids(
                        sampling_strategy={0: cluster_target}, 
                        random_state=42,
                        voting='soft'
                    )
                    
                    # Aplicar muestreo
                    features_resampled, y_resampled = cluster_centroids.fit_resample(
                        cluster_features_clean, y_artificial
                    )
                    
                    # Encontrar los índices más cercanos a los centroides
                    from sklearn.metrics.pairwise import euclidean_distances
                    
                    # Calcular distancias entre centroides y puntos originales
                    distances = euclidean_distances(features_resampled, cluster_features_clean)
                    selected_indices = []
                    
                    for i in range(len(features_resampled)):
                        # Encontrar el punto más cercano que no haya sido seleccionado
                        sorted_indices = np.argsort(distances[i])
                        for idx in sorted_indices:
                            if idx not in selected_indices:
                                selected_indices.append(idx)
                                break
                    
                    # Seleccionar registros correspondientes
                    cluster_sampled = cluster_df.iloc[selected_indices].copy()
                    
                except Exception as e:
                    print(f"   Warning: ClusterCentroids falló para cluster {cluster_id}, usando muestreo aleatorio")
                    cluster_sampled = cluster_df.sample(n=cluster_target, random_state=42)
            else:
                # Muestreo aleatorio simple para clusters pequeños
                cluster_sampled = cluster_df.sample(n=cluster_target, random_state=42)
            
            sampled_dfs.append(cluster_sampled)
            print(f"   Cluster {cluster_id}: {len(cluster_df)} → {len(cluster_sampled)} registros")
        
        # Combinar todos los clusters muestreados
        final_sample = pd.concat(sampled_dfs, ignore_index=True)
        
        # Ajuste final si no coincide exactamente con target_size
        if len(final_sample) != target_size:
            if len(final_sample) > target_size:
                final_sample = final_sample.sample(n=target_size, random_state=42)
            else:
                # Si es menor, completar con muestreo adicional
                remaining_needed = target_size - len(final_sample)
                if remaining_needed > 0:
                    available_df = df[~df.index.isin(final_sample.index)]
                    if len(available_df) >= remaining_needed:
                        additional = available_df.sample(n=remaining_needed, random_state=42)
                        final_sample = pd.concat([final_sample, additional], ignore_index=True)
        
        return final_sample.drop('geo_cluster', axis=1) if 'geo_cluster' in final_sample.columns else final_sample
    
    # Aplicar muestreo estratificado a ambos datasets
    print(f"\n   Aplicando muestreo estratificado...")
    yellow_advanced = stratified_cluster_sampling(df_yellow, yellow_features, target_size_per_class, "Yellow")
    green_advanced = stratified_cluster_sampling(df_green, green_features, target_size_per_class, "Green")
    
    # FASE 5: VERIFICACIÓN Y RESULTADOS
    print(f"\nFASE 5: VERIFICACIÓN DE RESULTADOS...")
    
    final_yellow_size = len(yellow_advanced)
    final_green_size = len(green_advanced)
    total_final = final_yellow_size + final_green_size
    
    yellow_final_pct = (final_yellow_size / total_final) * 100
    green_final_pct = (final_green_size / total_final) * 100
    
    print(f"RESULTADO FINAL:")
    print(f"   Yellow final: {final_yellow_size:,} registros ({yellow_final_pct:.2f}%)")
    print(f"   Green final:  {final_green_size:,} registros ({green_final_pct:.2f}%)")
    print(f"   Total final:  {total_final:,} registros")
    print(f"   Balance:      {final_yellow_size/final_green_size:.3f}:1 (Yellow:Green)")
    
    # Verificar balance 50/50
    balance_achieved = abs(yellow_final_pct - 50.0) < 1.0  # Tolerancia de 1%
    print(f"   Balance 50/50: {'✓ LOGRADO' if balance_achieved else '⚠ APROXIMADO'}")
    
    # Estadísticas de reducción
    yellow_reduction = total_yellow_orig - final_yellow_size
    green_reduction = total_green_orig - final_green_size
    total_reduction = (total_yellow_orig + total_green_orig) - total_final
    
    print(f"\nESTADÍSTICAS DE REDUCCIÓN:")
    print(f"   Yellow reducido en: {yellow_reduction:,} registros ({yellow_reduction/total_yellow_orig*100:.1f}%)")
    print(f"   Green reducido en:  {green_reduction:,} registros ({green_reduction/total_green_orig*100:.1f}%)")
    print(f"   Total reducido en:  {total_reduction:,} registros")
    
    print(f"\nVENTAJAS DEL MÉTODO AVANZADO:")
    print(f"   ✓ Balance perfecto 50/50")
    print(f"   ✓ Preserva diversidad geográfica mediante estratificación")
    print(f"   ✓ Usa ClusterCentroids para muestreo representativo")
    print(f"   ✓ Mantiene patrones espaciales y temporales")
    print(f"   ✓ Óptimo para algoritmos de Machine Learning")
    
    return yellow_advanced, green_advanced

def show_advanced_comparison(yellow_orig, green_orig, yellow_adv, green_adv):
    """Muestra comparación completa del método avanzado"""
    print("\n" + "=" * 100)
    print("COMPARACIÓN COMPLETA: ORIGINAL vs AVANZADO (CLUSTERCENTROIDS + ESTRATIFICACIÓN)")
    print("=" * 100)
    
    # Estadísticas originales
    total_orig = len(yellow_orig) + len(green_orig)
    yellow_orig_pct = len(yellow_orig) / total_orig * 100
    green_orig_pct = len(green_orig) / total_orig * 100
    
    # Estadísticas avanzadas
    total_adv = len(yellow_adv) + len(green_adv)
    yellow_adv_pct = len(yellow_adv) / total_adv * 100
    green_adv_pct = len(green_adv) / total_adv * 100
    
    # Tabla comparativa
    print(f"{'Método':<25} {'Total':<12} {'Yellow':<12} {'Green':<12} {'Yellow %':<10} {'Green %':<10} {'Balance':<10}")
    print("-" * 100)
    print(f"{'Original':<25} {total_orig:<12,} {len(yellow_orig):<12,} {len(green_orig):<12,} {yellow_orig_pct:<10.2f} {green_orig_pct:<10.2f} {'DESBALANCEADO':<10}")
    print(f"{'Avanzado':<25} {total_adv:<12,} {len(yellow_adv):<12,} {len(green_adv):<12,} {yellow_adv_pct:<10.2f} {green_adv_pct:<10.2f} {'BALANCEADO':<10}")
    
    print(f"\nMETODOLOGÍA APLICADA:")
    print(f"ClusterCentroids: Muestreo inteligente basado en centroides")
    print(f"Estratificación Geográfica: Preserva diversidad espacial")
    print(f"Balance 50/50: Ideal para clasificación")
    print(f"KMeans: Identificación de regiones representativas")
    print(f"StandardScaler: Normalización de features")

# EJECUCIÓN DEL MÉTODO AVANZADO
print("EJECUTANDO MUESTREO AVANZADO: CLUSTERCENTROIDS + ESTRATIFICACIÓN GEOGRÁFICA...")
yellow_balanced_advanced, green_complete_advanced = create_advanced_balanced_sample_50_50(
    yellow_trip_aligned, 
    green_trip_aligned
)

# Mostrar comparación completa
show_advanced_comparison(
    yellow_trip_aligned, green_trip_aligned,           # Originales
    yellow_balanced_advanced, green_complete_advanced  # Balanceados avanzados
)


ADVANCED BALANCED SAMPLING: CLUSTERCENTROIDS + ESTRATIFICACIÓN GEOGRÁFICA
EJECUTANDO MUESTREO AVANZADO: CLUSTERCENTROIDS + ESTRATIFICACIÓN GEOGRÁFICA...
INICIANDO MUESTREO AVANZADO CON CLUSTERCENTROIDS...
ANÁLISIS INICIAL:
   Yellow original: 3,898,963 registros
   Green original:  48,205 registros
   Ratio original:  80.9:1 (Yellow:Green)

OBJETIVO DE BALANCEO 50/50:
   Target por clase: 48,205 registros
   Total final:      96,410 registros

FASE 2: EXTRACCIÓN DE FEATURES GEOGRÁFICAS...
ANÁLISIS INICIAL:
   Yellow original: 3,898,963 registros
   Green original:  48,205 registros
   Ratio original:  80.9:1 (Yellow:Green)

OBJETIVO DE BALANCEO 50/50:
   Target por clase: 48,205 registros
   Total final:      96,410 registros

FASE 2: EXTRACCIÓN DE FEATURES GEOGRÁFICAS...
Features extraídas: 7 columnas
Yellow features: (3898963, 7)
Green features: (48205, 7)

FASE 3: ESTRATIFICACIÓN GEOGRÁFICA CON KMEANS...
Features extraídas: 7 columnas
Yellow features: (3898963, 7)
Green features: (

# Carga del Dataset Completo Trip Data  FHVHV 06-2025

In [6]:
# --- 1. CONFIGURACIÓN DEL PIPELINE  ---

# Llaves corregidas para que coincidan con los datos (Jul, Ago, Sep)
FHVHV_URLS = {
    "fhvhv_julio_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-07.parquet",
    "fhvhv_agosto_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-08.parquet",
    "fhvhv_septiembre_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-09.parquet"
}
# 
MODO_DESARROLLO = True  # Cambiar a False para producción

#### Carga y concatena múltiples archivos Parquet

In [7]:
# --- 2. LA FUNCIÓN DEL PIPELINE CON BALANCEADO EQUILIBRADO ---
def crear_dataset_final(fuentes_url: dict, modo_dev: bool = True, colores_fuentes: dict = None, equilibrar_fuentes: bool = True):
    """
    Carga y concatena múltiples archivos Parquet desde URLs o DataFrames con balanceado equilibrado.
    
    - Si modo_dev=True: Toma una muestra rápida del 1% de cada archivo.
    - Si modo_dev=False: Carga el 100% de cada archivo.
    - Si equilibrar_fuentes=True: Balancea el número de registros entre fuentes.
    - Si colores_fuentes está definido, añade la columna 'taxi_color' usando ese mapping.
    """
    lista_dataframes = []
    tamaños_originales = {}
    
    if modo_dev:
        print(f"--- MODO DESARROLLO (Muestra: 1%) ---")
        sample_fraction = 0.01  # 1%
    else:
        print(f"--- MODO PRODUCCIÓN (Datos: 100%) ---")
        sample_fraction = 1.0  # 100%
    
    if equilibrar_fuentes:
        print(f"MODO EQUILIBRADO ACTIVADO - Balanceando fuentes de datos")

    con = duckdb.connect(database=':memory:')
    
    # FASE 1: Cargar datos iniciales y obtener tamaños
    for nombre_fuente, fuente in fuentes_url.items():
        print(f"Procesando: {nombre_fuente}...")
        
        try:
            # Si es un DataFrame de pandas, lo usamos directamente
            if isinstance(fuente, pd.DataFrame):
                if modo_dev:
                    df_temp = fuente.sample(frac=sample_fraction, random_state=42)
                else:
                    df_temp = fuente.copy()
                print(f"-> Cargado desde DataFrame: {len(df_temp)} filas")
            
            # Si es una URL, usamos DuckDB para cargarlo
            elif isinstance(fuente, str):
                if modo_dev:
                    query = f"""
                    SELECT *
                    FROM '{fuente}'
                    USING SAMPLE {int(sample_fraction * 100)}%
                    """
                else:
                    query = f"""
                    SELECT *
                    FROM '{fuente}'
                    """
                
                # Ejecutar la consulta y convertir a DataFrame
                df_temp = con.execute(query).df()
                print(f"-> Cargado desde URL: {len(df_temp)} filas")
            
            else:
                print(f"-> ERROR: Tipo de fuente no soportado para {nombre_fuente}")
                continue
            
            # Añadir metadatos de procedencia
            if not df_temp.empty:
                df_temp = df_temp.copy()
                df_temp['fuente_origen'] = nombre_fuente
                lista_dataframes.append(df_temp)
                tamaños_originales[nombre_fuente] = len(df_temp)
            else:
                print(f"-> Advertencia: {nombre_fuente} está vacío")
                
        except Exception as e:
            print(f"-> ERROR procesando {nombre_fuente}: {e}")
            continue
    
    con.close()

    print(f"\nConcatenando DataFrames equilibrados...")
    df_final = pd.concat(lista_dataframes, axis=0, ignore_index=True)
    
    # Mezclar el DataFrame final para distribuir las fuentes
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"¡Proceso completado! Total de filas: {len(df_final):,}")
    
    # Mostrar distribución final por fuente
    if 'fuente_origen' in df_final.columns:
        print(f"\nDistribución final por fuente:")
        distribucion_final = df_final['fuente_origen'].value_counts()
        for fuente, cantidad in distribucion_final.items():
            porcentaje = (cantidad / len(df_final)) * 100
            print(f"   • {fuente}: {cantidad:,} registros ({porcentaje:.1f}%)")
    
    return df_final

# Creación de Muestra 1% del Dataset FHVHV - Taxi Verde & Taxi Amarillo

In [8]:
# --- 3. EJECUCIÓN DEL PIPELINE ---

print("=== Iniciando Pipeline SOLO para FHVHV (Uber/Lyft) ===")
df_fhvhv_sample = crear_dataset_final(FHVHV_URLS, modo_dev=MODO_DESARROLLO)

print("\n" + "="*60 + "\n")

print("=== Dataset TAXIS (Yellow/Green) - BALANCEADO AVANZADO ===")
print("Usando método ClusterCentroids + Estratificación Geográfica:")
print(f"• Yellow avanzado: {len(yellow_balanced_advanced):,} registros")
print(f"• Green avanzado:  {len(green_complete_advanced):,} registros")

# Concatenar datasets balanceados con método avanzado
df_taxis_equilibrado = pd.concat([yellow_balanced_advanced, green_complete_advanced], ignore_index=True)
df_taxis_equilibrado = df_taxis_equilibrado.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"• Taxis equilibrados totales: {len(df_taxis_equilibrado):,} registros")

print("\n" + "="*60 + "\n")
print("Procesamiento finalizado:")
print(f"1. df_fhvhv_sample: {len(df_fhvhv_sample):,} filas (pipeline)")
print(f"2. df_taxis_equilibrado: {len(df_taxis_equilibrado):,} filas (equilibrado directo)")

=== Iniciando Pipeline SOLO para FHVHV (Uber/Lyft) ===
--- MODO DESARROLLO (Muestra: 1%) ---
MODO EQUILIBRADO ACTIVADO - Balanceando fuentes de datos
Procesando: fhvhv_julio_2025...
-> Cargado desde URL: 182272 filas
Procesando: fhvhv_agosto_2025...
-> Cargado desde URL: 182272 filas
Procesando: fhvhv_agosto_2025...
-> Cargado desde URL: 167936 filas
Procesando: fhvhv_septiembre_2025...
-> Cargado desde URL: 167936 filas
Procesando: fhvhv_septiembre_2025...
-> Cargado desde URL: 157696 filas

Concatenando DataFrames equilibrados...
-> Cargado desde URL: 157696 filas

Concatenando DataFrames equilibrados...
¡Proceso completado! Total de filas: 507,904

Distribución final por fuente:
   • fhvhv_julio_2025: 182,272 registros (35.9%)
   • fhvhv_agosto_2025: 167,936 registros (33.1%)
   • fhvhv_septiembre_2025: 157,696 registros (31.0%)


=== Dataset TAXIS (Yellow/Green) - BALANCEADO AVANZADO ===
Usando método ClusterCentroids + Estratificación Geográfica:
• Yellow avanzado: 48,205 registro

# Carga del Dataset FHVHV (Uber/Lyft)  & TAXIS (Yellow/Green)

### FHVHV (Uber/Lyft)

In [9]:
# FHVHV (Uber/Lyft)
print("Guardando muestra de FHVHV (Uber/Lyft)...")
archivo_fhvhv = 'fhvhv_trimestral.parquet'

try:
    df_fhvhv_sample.to_parquet(archivo_fhvhv, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_fhvhv}'")
except Exception as e:
    print(f"-> ERROR al guardar FHVHV: {e}")

Guardando muestra de FHVHV (Uber/Lyft)...
-> ¡Éxito! Guardado como 'fhvhv_trimestral.parquet'
-> ¡Éxito! Guardado como 'fhvhv_trimestral.parquet'


### TAXIS (Yellow/Green)


In [10]:
# TAXIS (Yellow/Green) - Dataset Equilibrado Directo
print("\nGuardando dataset EQUILIBRADO de TAXIS (Yellow/Green)...")
archivo_taxis_equilibrado = 'taxis_equilibrado_directo.parquet'

try:
    df_taxis_equilibrado.to_parquet(archivo_taxis_equilibrado, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_taxis_equilibrado}'")
    print(f"-> Total registros: {len(df_taxis_equilibrado):,}")
    
    # Verificar distribución
    if 'taxi_color' in df_taxis_equilibrado.columns:
        distribucion = df_taxis_equilibrado['taxi_color'].value_counts()
        print(f"-> Distribución por color:")
        for color, cantidad in distribucion.items():
            porcentaje = (cantidad / len(df_taxis_equilibrado)) * 100
            print(f"   • {color.title()}: {cantidad:,} ({porcentaje:.1f}%)")
            
except Exception as e:
    print(f"-> ERROR al guardar Taxis: {e}")


Guardando dataset EQUILIBRADO de TAXIS (Yellow/Green)...
-> ¡Éxito! Guardado como 'taxis_equilibrado_directo.parquet'
-> Total registros: 96,410
-> Distribución por color:
   • Green: 48,205 (50.0%)
   • Yellow: 48,205 (50.0%)


### Carga de Datos Procesados

In [11]:
# Cargar datasets procesados
taxis_equilibrado_final = pd.read_parquet('taxis_equilibrado_directo.parquet')
fhvhv_sample = pd.read_parquet('fhvhv_trimestral.parquet')

print("Datasets cargados:")
print(f"• Taxis equilibrados: {len(taxis_equilibrado_final):,} registros")
print(f"• FHVHV sample: {len(fhvhv_sample):,} registros")

Datasets cargados:
• Taxis equilibrados: 96,410 registros
• FHVHV sample: 507,904 registros
